# Grupo A · Cheios, vazios e uso do solo
**Oficina Python para Mapeamento Urbano** · Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/))

Vocês vão comparar o **tecido urbano** de dois ou três recortes de BH usando as edificações e o uso do solo do OpenStreetMap, organizados numa grade regular de 100 m × 100 m.

### O que o grupo entrega (60 min)

| Tempo | Etapa | Produto |
|---|---|---|
| 10 min | Pergunta e hipótese | H1 e H0 escritas na seção 1 |
| 15 min | Conhecer e limpar o dado | tipos corrigidos, vazios tratados, recorte definido |
| 20 min | Um método **espacial** + um método **quantitativo** | tabela de resultados |
| 15 min | Mapa + gráfico com créditos | `outputs/grupo_X_mapa.png` e `outputs/grupo_X_grafico.png` |

No fim, cada grupo apresenta em 6 minutos: pergunta, método, mapa, gráfico e **um limite** da análise.

## 0. Ambiente

In [ ]:
!pip install -q geobr osmnx mapclassify folium
import os, sys
REPO = "https://github.com/brfelipealmeida/oficina-python-mapeamento"
if not os.path.exists("oficina-python-mapeamento"):
    !git clone -q {REPO}
sys.path.append("oficina-python-mapeamento/src")

import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from estilo import *
from dados import *
configurar_estilo()

## 1. Pergunta e hipótese

Escolham **uma** e reescrevam com as palavras do grupo (ou criem outra):

- O Centro tem taxa de ocupação maior que o Barreiro e Venda Nova?
- Quanto mais perto de uma estação de metrô, maior a ocupação do solo?
- Onde há mais área verde, há menos área construída? (parece óbvio: o dado confirma?)

**H1:** _escreva aqui_
**H0:** _escreva aqui_
**Variável dependente:** _..._  **Variável independente:** _..._

## 2. Carregar os dados (pronto)

Recortes circulares de 1 km de raio. Vocês podem trocar ou acrescentar pontos (lat, lon): basta copiar do Google Maps.

In [ ]:
RECORTES = {
    "Centro":      (-19.9192, -43.9386),
    "Barreiro":    (-19.9765, -44.0250),
    "Venda Nova":  (-19.8170, -43.9550),
}
RAIO = 1000

edificacoes, usos, circulos = [], [], []
for nome, ponto in RECORTES.items():
    circ = gpd.GeoSeries(gpd.points_from_xy([ponto[1]], [ponto[0]]), crs=CRS_WEB).to_crs(CRS_BH).buffer(RAIO)
    e = osm_feicoes({"building": True}, ponto=ponto, raio=RAIO)
    u = osm_feicoes({"landuse": True, "leisure": ["park", "garden"]}, ponto=ponto, raio=RAIO)
    for gdf, lista in ((e, edificacoes), (u, usos)):
        gdf = gdf.clip(circ).explode(index_parts=False)
        gdf = gdf[gdf.geom_type == "Polygon"].copy()   # recorte pode gerar linhas soltas
        gdf["recorte"] = nome
        lista.append(gdf)
    circulos.append(gpd.GeoDataFrame({"recorte": [nome]}, geometry=circ, crs=CRS_BH))

edif = pd.concat(edificacoes, ignore_index=True)
uso = pd.concat(usos, ignore_index=True)
circulos = pd.concat(circulos, ignore_index=True)
est = metro_bh()
print(edif.groupby("recorte").size())

## 3. Conhecer e limpar

O OSM traz dezenas de colunas, a maioria vazia. Perguntas para o grupo:
1. Quais colunas interessam? (`building`, `building:levels`, `landuse`, `leisure`, `name`)
2. Qual o tipo de `building:levels`? Por que não é número?
3. Quantas edificações têm número de pavimentos preenchido em cada recorte? Dá para usar?

In [ ]:
edif[["recorte", "building", "building:levels"]].head() if "building:levels" in edif else edif[["recorte", "building"]].head()

In [ ]:
# TODO: converter pavimentos para número e medir a cobertura do campo
if "building:levels" in edif:
    edif["pavimentos"] = pd.to_numeric(edif["building:levels"], errors="coerce")
    print(edif.groupby("recorte")["pavimentos"].apply(lambda s: s.notna().mean()).round(2))

### A grade de análise (pronta)

Mapas por polígonos irregulares dificultam a comparação. Uma grade regular transforma cada recorte em células comparáveis.

In [ ]:
from shapely.geometry import box

def grade(area, lado=100):
    x0, y0, x1, y1 = area.total_bounds
    cel = [box(x, y, x + lado, y + lado) for x in np.arange(x0, x1, lado) for y in np.arange(y0, y1, lado)]
    g = gpd.GeoDataFrame(geometry=cel, crs=area.crs)
    return gpd.sjoin(g, area[["recorte", "geometry"]], predicate="within").drop(columns="index_right")

celulas = grade(circulos).reset_index(drop=True)
celulas["id"] = celulas.index

# Área construída em cada célula (overlay = interseção geométrica)
pedacos = gpd.overlay(edif[["geometry"]], celulas[["id", "geometry"]], how="intersection")
pedacos["a"] = pedacos.area
celulas["taxa_ocupacao"] = celulas["id"].map(pedacos.groupby("id")["a"].sum()).fillna(0) / celulas.area
celulas.groupby("recorte")["taxa_ocupacao"].describe().round(2)

## 4. Método espacial (escolha um e complete)

**a) Distância ao metrô** (`sjoin_nearest`): cada célula recebe a distância até a estação mais próxima.
**b) Área verde por célula** (`overlay`): mesma lógica da taxa de ocupação, com `uso` filtrado por parques e gramados.
**c) Buffer**: classificar as células em "até 500 m da estação" e "além de 500 m".

In [ ]:
# a) distância à estação mais próxima (já pronto, adaptem se escolherem b ou c)
cent = celulas.copy()
cent["geometry"] = celulas.centroid
cent = gpd.sjoin_nearest(cent, est[["estacao", "geometry"]], distance_col="dist_metro_m")
celulas["dist_metro_m"] = cent.groupby("id")["dist_metro_m"].first()
celulas["perto_metro"] = celulas["dist_metro_m"] <= 500
celulas.groupby("recorte")[["dist_metro_m"]].median()

In [ ]:
# b) TODO: área verde por célula
# verde = uso[uso["leisure"].isin(["park", "garden"]) | uso["landuse"].isin(["grass", "forest"])]
# ...

## 5. Método quantitativo (escolha um)

| Opção | Quando usar | Código |
|---|---|---|
| Correlação | duas variáveis numéricas | `df["a"].corr(df["b"], method="spearman")` |
| Regressão | quanto Y muda quando X muda, com controles | `smf.ols("y ~ x + controle", data=df).fit()` |
| Comparação de grupos | uma variável numérica entre categorias | `stats.mannwhitneyu(g1, g2)` |

Lembre: p-valor pequeno indica que a diferença dificilmente é acaso; **não** indica que ela é grande ou importante. Olhe também o tamanho do efeito (diferença de medianas, coeficiente).

In [ ]:
# Exemplo: comparar a ocupação entre dois recortes
g1 = celulas.loc[celulas.recorte == "Centro", "taxa_ocupacao"]
g2 = celulas.loc[celulas.recorte == "Barreiro", "taxa_ocupacao"]
print("Medianas:", round(g1.median(), 2), "vs", round(g2.median(), 2))
print(stats.mannwhitneyu(g1, g2))

In [ ]:
# TODO: correlação ou regressão (ex.: taxa_ocupacao ~ dist_metro_m + recorte)

## 6. Mapa (complete títulos e escolha a variável)

In [ ]:
fig, axs = plt.subplots(1, len(RECORTES), figsize=(5 * len(RECORTES), 5.5))
for ax, (nome, g) in zip(axs, celulas.groupby("recorte")):
    ax.set_axis_off(); ax.set_aspect("equal")
    g.plot(ax=ax, column="taxa_ocupacao", cmap=CMAP, vmin=0, vmax=0.8, linewidth=0)
    edif[edif.recorte == nome].plot(ax=ax, color=CORES["tinta"], linewidth=0, alpha=0.25)
    ax.set_title(nome, loc="left", fontsize=14)
    barra_escala(ax, 500)
fig.suptitle("TODO: título que responde à pergunta", x=0.02, ha="left", fontsize=20,
             fontweight="semibold", fontfamily="Barlow Condensed")
creditos(fig, fonte="OpenStreetMap", nota="Grupo A")
salvar(fig, "grupo_A_mapa")

## 7. Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ordem = list(RECORTES)
ax.boxplot([celulas.loc[celulas.recorte == r, "taxa_ocupacao"] for r in ordem], tick_labels=ordem,
           patch_artist=True, boxprops={"facecolor": CORES["lilas"]}, medianprops={"color": CORES["roxo"]})
ax.set_ylabel("Taxa de ocupação da célula (0 a 1)")
titulo(ax, "TODO: título", "Células de 100 m × 100 m")
creditos(fig, fonte="OpenStreetMap", nota="Grupo A")
salvar(fig, "grupo_A_grafico")

## 8. Conclusão (escreva aqui)

1. **Resultado:** a hipótese foi sustentada, refutada ou ficou inconclusiva? Com que número?
2. **Leitura urbana:** o que isso significa para quem planeja a cidade?
3. **Limite:** o que o dado não mostra? (cobertura do OSM, escala do setor, distância em linha reta vs. caminho real, relevo...)